# Procesamiento de archivos de registros

## Introducción

Una estación registra el nivel de agua de un pozo y almacena las mediciones en el archivo <a href="../niveles_ancho_fijo.txt" download="niveles_ancho_fijo.txt">`niveles_ancho_fijo.txt`</a> . Cada línea del archivo contiene la fecha y el nivel de agua de una medición:

```text
24-01-201222.700
19-03-201222.560
16-05-201223.290
.
:
```

El formato tiene **ancho fijo**: los primeros 10 caracteres representan la fecha y los siguientes 6, el nivel en metros.

La siguiente función recorre línea por línea del archivo, y extrae la fecha de los primeros 10 caracteres y convierte los siguientes 6 en un número real que representa el nivel de agua. Finalmente, retorna una lista de registros, cada uno compuesto por la fecha y el nivel correspondiente.

In [20]:
def leer_niveles(nombre_archivo):

    fechas, niveles = [], []

    try:
        fid = open(nombre_archivo)
    except FileNotFoundError:
        print("Archivo no encontrado!")
        return fechas, niveles

    for linea in fid:
        fecha, nivel = linea[0:10], float(linea[10:16])
        fechas.append(fecha)
        niveles.append(nivel)

    fid.close()

    return fechas, niveles

fechas, niveles = leer_niveles("./datos/niveles_ancho_fijo.txt")

Análice lo siguiente,
- ¿Qué ocurrirá si la fecha cambiara de longitud?
- ¿Y si se agrega como valor el nombre de la estación de longitud variable?
- ¿Cómo el programa puede determinar la longitud de cada dato?

## Registro y campos

<img src="../../assets/images/fundamentos-programacion/cs_data_hierarchy.png"/>

Un **registro** es una colección de campos lógicamente relacionados que pueden tratarse como una unidad. En general son usados para describir entidades, por ejemplo, una persona, una venta o una observación, una medición, etc.

Un **campo** es un elemento de datos con significado dentro del registro. Su interpretación incluye un tipo esperado, una unidad cuando corresponde y reglas de validez.

Un **archivo de registros** reúne registros que siguen un esquema común.

## Archivos CSV

CSV es un **formato de texto** utilizado para intercambiar datos tabulares. La RFC 4180 [@shafranovich2005csv] documenta el formato separado por comas (`,`). En la práctica también se usan variantes con punto y coma (`;`) o tabulación.

- Un **delimitador** separa los campos de cada registro.
- Puede existir un **encabezado** con los nombres de las columnas.
- Las **comillas** permiten incluir delimitadores, comillas y saltos de línea dentro de un campo.
- CSV tiene estructura tabular, pero no impone por sí mismo tipos, campos obligatorios ni reglas de validez.

El módulo [`csv`](https://docs.python.org/es/3/library/csv.html) permite leer y escribir estos formatos sin separar manualmente las cadenas.

## Lectura de registros con `reader`

- `reader()` crea un objeto iterable (puede recorrerse con un ciclo `for`).
- En cada iteración entrega un registro como **lista de cadenas**.
- Incluye el encabezado en la iteración.

In [5]:
from csv import reader

fid = open('./registros/niveles.csv')

registros = reader(fid)
for registro in registros:
    print(registro)

fid.close()

['fecha;nivel']
['24-01-2012;22.700']
['19-03-2012;22.560']
['16-05-2012;23.290']
['16-01-2013;25.330']
['09-09-2013;24.060']
['20-11-2013;24.420']
['15-01-2014;25.210']
['19-03-2014;23.850']
['19-11-2014;19.950']
['14-01-2015;26.760']
['20-05-2015;24.630']


La configuración anterior es incorrecta para este archivo:
- entrega un único campo por registro;
- el parámetro `delimiter` permite establecer el separador correcto.

In [6]:
from csv import reader

fid = open('./registros/niveles.csv')

registros = reader(fid, delimiter=';')
for registro in registros:
    print(registro)

fid.close()

['fecha', 'nivel']
['24-01-2012', '22.700']
['19-03-2012', '22.560']
['16-05-2012', '23.290']
['16-01-2013', '25.330']
['09-09-2013', '24.060']
['20-11-2013', '24.420']
['15-01-2014', '25.210']
['19-03-2014', '23.850']
['19-11-2014', '19.950']
['14-01-2015', '26.760']
['20-05-2015', '24.630']


::::{hint} Ejercicio.

Diseñe una función que lea los registros del archivo `niveles_coma_decimal.csv` y retorne dos listas del mismo tamaño. La primera con las fechas como cadenas de texto y la segunda con los niveles como números flotantes. Los elementos ubicados en la misma posición de ambas listas deben corresponder a una misma medición.

:::{dropdown} Solución.

```python
from csv import reader


def leer_niveles(archivo, simbolo_decimal=','):

    fechas, niveles = [], []

    try:
        fid = open(archivo)
    except FileNotFoundError:
        print('Archivo no encontrado!')
        return fechas, niveles

    registros = reader(fid, delimiter=';')

    encabezado = True
    for registro in registros:
        if encabezado:
            encabezado = False
            continue

        fecha = registro[0]
        nivel = registro[1]

        if simbolo_decimal in nivel:
            nivel = convierte_a_decimal(nivel, simbolo_decimal)

        fechas.append(fecha)
        niveles.append(nivel)

    fid.close()

    return fechas, niveles


def convierte_a_decimal(campo, simbolo):

    idx = campo.index(simbolo)
    decimal = campo[:idx] + '.' + campo[idx+1:]

    return float(decimal)
```

:::

::::

## Lectura de registros como diccionarios

- `DictReader()` crea un lector que asocia cada nombre de campo con su valor.
- Si el archivo no tiene encabezado, se pueden proporcionar los nombres mediante el parámetro `fieldnames`.
- Al proporcionar `fieldnames`, el lector utiliza el primer registro como encabezado y no lo devuelve como dato, y permite consultar los nombres de los campos.

In [10]:
from csv import DictReader

fid = open('./registros/niveles_sin_encabezado.csv')
registros = DictReader(fid, delimiter=';', fieldnames=['fecha', 'nivel'])

for registro in registros:
    print(registro['fecha'], registro['nivel'])

fid.close()

24-01-2012 22.700
19-03-2012 22.560
16-05-2012 23.290
16-01-2013 25.330
09-09-2013 24.060
20-11-2013 24.420
15-01-2014 25.210
19-03-2014 23.850
19-11-2014 19.950
14-01-2015 26.760
20-05-2015 24.630


Por defecto, `DictReader`
- Sustituye los valores ausentes por `None`.
- Si sobran, se almacenan en una lista asociada a la clave `None`.

In [19]:
from csv import DictReader

fid = open('./registros/niveles_revision.csv')
registros = DictReader(fid, delimiter=';')
for registro in registros:
    print(registro)

fid.close()

{'fecha': '24-01-2012', 'nivel': '22.700'}
{'fecha': '19-03-2012', 'nivel': '22.560'}
{'fecha': '16-05-2012', 'nivel': '23.290'}
{'fecha': '16-01-2013', 'nivel': '25.330'}
{'fecha': '09-09-2013', 'nivel': '24.060'}
{'fecha': '20-11-2013', 'nivel': '24.420'}
{'fecha': '15-01-2014', 'nivel': '25.210'}
{'fecha': '19-03-2014', 'nivel': ''}
{'fecha': '19-11-2014', 'nivel': 'sin dato'}
{'fecha': '14-01-2015', 'nivel': '-1.0'}
{'fecha': '20-05-2015', 'nivel': None}
{'fecha': '21-05-2015', 'nivel': '24.0', None: ['extra']}
{'fecha': '', 'nivel': '23.0'}


## Escritura a partir de listas

- `writer()` crea un objeto escritor que convierte los valores de un registro a la representación CSV acordada. **No debe confundirse con `write()`, que escribe texto directamente en un archivo**.
- `writerow()` escribe un registro a partir de sus campos.
- `writerows()` escribe una colección de registros.

Recordar que,
- el modo `'w'` crea el archivo o sobreescribe su contenido.
- el modo `'a'` agrega registros al final.

In [31]:
from csv import writer

def agregar_persona(usuario, fecha):
    fid = open('./registros/log_usuario.csv', 'a')

    escritor = writer(fid, delimiter=';')
    escritor.writerow([fecha, usuario])

    fid.close()

    print('Acceso agregado a log_usuario.csv.')


agregar_persona(usuario='Juanito', fecha='19-11-2026')

Acceso agregado a log_usuario.csv.


En el siguiente ejemplo, se define una función para agregar una serie de registros.
- Al escribir, el escritor representa los números como texto.

In [30]:
from csv import writer

def agregar_personas(personas):

    fid = open('./registros/personas.csv', 'w')

    escritor = writer(fid, delimiter=',')
    escritor.writerow(['nombre', 'edad', 'genero'])
    escritor.writerows(personas)

    fid.close()

    print('Archivo personas.csv escrito con encabezado.')


agregar_personas(
    [
        ['María Pérez', 18, 'F'],
        ['Juan Pino', 25, 'M'],
        ['Roberto Pinto', 17, 'M']
    ]
)

Archivo personas.csv escrito con encabezado.


## Escritura a partir de diccionarios

`DictWriter()` permite escribir archivos CSV a partir de diccionarios de Python, asociando las claves de cada diccionario con las columnas del archivo.

- Recibe el archivo abierto en modo escritura y el parámetro `fieldnames`, que establece los nombres y el orden de las columnas.
- Opcionalmente, permite especificar `delimiter` para definir el carácter que separa los valores (por defecto, una coma).
- `writeheader()` escribe la primera fila del archivo con los nombres de las columnas definidos en `fieldnames`.
- `writerow()` escribe una fila a partir de un diccionario, colocando cada valor en la columna correspondiente a su clave.
- `writerows()` permite escribir varias filas a partir de una lista de diccionarios.

In [24]:
from csv import DictWriter

personas = [
    {'nombre': 'María Pérez', 'edad': 18, 'genero': 'F'},
    {'nombre': 'Juan Pino', 'edad': 25, 'genero': 'M'},
    {'nombre': 'Roberto Pinto', 'edad': 17, 'genero': 'M'}
]

encabezado = ['nombre', 'edad', 'genero']

fid = open('./registros/personas_2.csv', 'w')

escritor = DictWriter(fid, delimiter=';', fieldnames=encabezado)
escritor.writeheader()
for persona in personas:
    escritor.writerow(persona)

fid.close()

- Los diccionarios pueden contener claves diferentes entre sí, siempre que estén definidas en `fieldnames`. Si un diccionario contiene claves que no están definidas en `fieldnames`, DictWriter() genera un `ValueError` al intentar escribirlo, por defecto.
- Las claves ausentes generan celdas vacías, mientras que las claves adicionales producen un error por defecto.
- El orden de las columnas siempre está determinado por `fieldnames`, independientemente del orden de las claves en cada diccionario.

:::{hint} Ejercicio.

El archivo `niveles_revision.csv` contiene mediciones del nivel de agua de un pozo. Utiliza punto y coma (`;`) como delimitador y su primera fila corresponde al encabezado `fecha;nivel`. Algunos registros presentan datos incompletos o incorrectos.

Desarrolle un programa que lea el archivo, valide sus registros y genere un informe de las mediciones que requieren atención. Considere válido un registro cuando tenga únicamente dos campos, ambos sean distintos de una cadena vacía y el nivel pueda convertirse a un número flotante comprendido entre 0 y 30 metros, inclusive. No es necesario validar el formato de la fecha.

El programa debe:

1. **Leer y clasificar los registros.** Obtener la cantidad de registros leídos, aceptados y rechazados, sin contar el encabezado. Cada registro debe clasificarse una sola vez. Para los rechazados, muestre el motivo del rechazo.
2. **Calcular el promedio.** Obtener el promedio de los niveles de las mediciones aceptadas. Si no existen mediciones válidas, informe esta situación.
3. **Identificar las alertas.** Contar las mediciones válidas cuyo nivel supere los 24 metros.
4. **Generar el archivo de alertas.** Crear `alertas.csv`, utilizando punto y coma como delimitador y el encabezado `fecha;nivel`. Debe incluir únicamente la fecha y el nivel de las mediciones válidas que generan una alerta. Si no hay alertas, el archivo debe contener solo el encabezado.
5. **Verificar los resultados.** Leer `alertas.csv` y comprobar que su cantidad de registros de datos coincide con el contador de alertas. Verificar además que la cantidad de registros leídos del archivo original sea igual a la suma de los válidos y rechazados.

Al finalizar, debe mostrar las cuentas, el promedio cuando corresponda y el resultado de ambas verificaciones. El archivo original debe conservarse sin modificaciones.